# CattleSense AI — Research Prototype v1.1

**Goal:** Evaluate a cattle-activity Random Forest on **six held-out animals**, one at a time, and understand failures in rumination (RUS). This is **evaluation research**, not a claim of improved performance or proof of novelty.

- Dataset: [Japanese Black Beef Cow Behavior Classification Dataset v2](https://zenodo.org/records/5849025)
- Classes: `RES`, `RUS`, `MOV`, `GRZ`
- 25 Hz accelerometer, 50-sample windows, 25-sample stride.
- Do **not** upload the source dataset to public GitHub; check the original CC BY-NC-ND terms.
- **This notebook does not replace or modify your existing website/model.**

Run **Runtime → Run all** once. If the dataset download fails, upload the original `JapaneseBlackBeefData.zip` into Colab Files and run again.

## 1 — Libraries and output folder

In [ ]:
import re, zipfile, urllib.request, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display

SEED=42
np.random.seed(SEED)
OUT=Path('cattlesense_v11_results'); OUT.mkdir(exist_ok=True)
KEEP=['RES','RUS','MOV','GRZ']
print('Ready. Results will be saved to', OUT.resolve())

## 2 — Download the six-cow dataset

In [ ]:
DATA_URL='https://zenodo.org/records/5849025/files/JapaneseBlackBeefData.zip?download=1'
zip_path=Path('JapaneseBlackBeefData.zip')
if not zip_path.exists():
    print('Downloading official ZIP from Zenodo...')
    try:
        urllib.request.urlretrieve(DATA_URL, zip_path)
    except Exception as exc:
        raise RuntimeError('Download failed. Upload JapaneseBlackBeefData.zip manually using the Colab Files sidebar, then retry.') from exc
if not zipfile.is_zipfile(zip_path):
    raise ValueError('The downloaded file is not a valid ZIP. Delete it and download the dataset from Zenodo.')
with zipfile.ZipFile(zip_path) as z:
    csv_names=[name for name in z.namelist() if re.search(r'(?:^|/)cow[1-6]\.csv$',name,re.I)]
assert len(csv_names)==6, f'Expected six cow CSVs; found {csv_names}'
print('ZIP size (MB):', round(zip_path.stat().st_size/1e6,2))
print('Data CSVs:',csv_names)

## 3 — Read dataset and automatically fix timestamps

The data stores UNIX timestamps in **milliseconds** (about 40 between samples), but the gap checker expects **seconds**. Conversion is performed exactly once during load. No extra manual code cells needed.

In [ ]:
def normalized(s): return re.sub(r'[^a-z0-9]','',str(s).lower())

def load_cow(file_handle, cow_id):
    df=pd.read_csv(file_handle,low_memory=False)
    norms={normalized(c):c for c in df.columns}
    needed={'x':'accx','y':'accy','z':'accz','label':'label','time':'timestampunix'}
    mapping={new:next((orig for k,orig in norms.items() if k.startswith(prefix)),None) for new,prefix in needed.items()}
    if any(v is None for v in mapping.values()):
        raise ValueError(f'Unrecognized columns in cow {cow_id}: {list(df.columns)}')
    df=df.rename(columns={v:k for k,v in mapping.items()})[['time','x','y','z','label']].copy()
    for col in ['time','x','y','z']:
        df[col]=pd.to_numeric(df[col],errors='coerce')
    # Dataset UNIX timestamps are milliseconds. Subtract origin first to avoid float precision loss.
    df['time']=(df['time'].astype('float64')-float(df['time'].iloc[0]))/1000.0
    df['label']=df['label'].astype(str).str.strip().str.upper()
    df['cow']=cow_id
    return df

all_cows={}
with zipfile.ZipFile(zip_path) as z:
    for name in csv_names:
        cow=int(re.search(r'cow([1-6])\.csv$',name,re.I).group(1))
        with z.open(name) as f: all_cows[cow]=load_cow(f,cow)
print('Raw samples / cow:',{c:len(d) for c,d in all_cows.items()})
print('All labels:',sorted(set().union(*(set(d.label.unique()) for d in all_cows.values()))))
for cow,df in sorted(all_cows.items()):
    diffs=df.time.diff().dropna()
    positive=diffs[diffs>0]
    print(f'Cow {cow}: positive timestamp delta (median) = {positive.median() if len(positive) else "unavailable"}')

## 4 — Count original labels by animal

Different cows have different recorded classes. Missing classes in a test cow must be reported, not silently interpreted as good performance.

In [ ]:
dist=pd.DataFrame({f'cow_{k}':v.label.value_counts() for k,v in all_cows.items()}).fillna(0).astype(int)
display(dist.reindex(KEEP,fill_value=0))
ax=dist.reindex(KEEP,fill_value=0).T.plot.bar(figsize=(9,4),title='Number of labeled raw samples by cow')
ax.set_ylabel('Samples'); plt.tight_layout(); plt.savefig(OUT/'class_distribution.png',dpi=160); plt.show()

## 5 — Build valid two-second windows

Each window must have one label, finite sensor readings, and contiguous timestamps. This uses the same 17 features as the original website model.

In [ ]:
def features(a):
    a=np.asarray(a,dtype=np.float32)
    mag=np.linalg.norm(a,axis=1)
    return np.r_[a.mean(axis=0), a.std(axis=0), a.min(axis=0), a.max(axis=0), (a*a).mean(axis=0),mag.mean(),mag.std()].astype(np.float32)

def make_windows(df,width=50,stride=25,max_gap_seconds=0.12):
    a=df[['x','y','z']].to_numpy(dtype=np.float32)
    times=df.time.to_numpy(dtype=np.float64)
    labels=df.label.to_numpy()
    xs,ys=[],[]; rejected={'label':0,'missing':0,'time_gap':0}
    for start in range(0,len(df)-width+1,stride):
        target=labels[start:start+width]; data=a[start:start+width]; t=times[start:start+width]
        if target[0] not in KEEP or not np.all(target==target[0]): rejected['label']+=1; continue
        if not (np.isfinite(data).all() and np.isfinite(t).all()): rejected['missing']+=1; continue
        dt=np.diff(t)
        if np.any(dt<=0) or np.any(dt>max_gap_seconds): rejected['time_gap']+=1; continue
        xs.append(features(data)); ys.append(target[0])
    X=np.asarray(xs,dtype=np.float32).reshape(-1,17)
    return X,np.asarray(ys),rejected

windows={}
for cow,df in sorted(all_cows.items()):
    X,y,rej=make_windows(df)
    windows[cow]=(X,y)
    print(f'Cow {cow}: {len(y)} valid windows; rejected={rej}; labels={dict(pd.Series(y).value_counts())}')
if not any(len(y) for _,y in windows.values()):
    raise ValueError('No valid windows: inspect UNIX timestamp units/precision and adjust only after examining raw timestamp differences.')

## 6 — Leave-one-cow-out evaluation (six rounds)

For each round, choose one cow as **test**. From the remaining five animals, reserve one for **validation**, fit the Random Forest on four cows. Use validation only for selecting confidence threshold. The tested cow never appears in training or threshold selection. All settings are fixed in advance.

Accuracy and macro F1 are reported with all four target labels. **A class absent from test data will have undefined recall**, recorded as null in class metrics; macro F1 uses `zero_division=0` and should always be interpreted alongside label support.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

ALL_COWS=sorted(windows)
LABELS=KEEP
THRESHOLDS=np.linspace(0.30,0.99,140)
TARGET_ACC=0.85
MIN_ACCEPTED=30
records=[]
per_class=[]
confusions=[]
all_predictions=[]

def assemble(ids):
    return np.concatenate([windows[i][0] for i in ids]),np.concatenate([windows[i][1] for i in ids])

for test_cow in ALL_COWS:
    others=[c for c in ALL_COWS if c!=test_cow]
    # Deterministic rotation: use the next cow numerically as validation.
    val_cow=others[0]
    train_cows=[c for c in others if c!=val_cow]
    Xtr,ytr=assemble(train_cows)
    Xval,yval=assemble([val_cow])
    Xt,yt=assemble([test_cow])
    if min(len(ytr),len(yval),len(yt))==0:
        raise ValueError(f'Empty split in test-cow {test_cow} round')
    rf=RandomForestClassifier(n_estimators=150,max_depth=18,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1)
    rf.fit(Xtr,ytr)
    pred=rf.predict(Xt)
    acc=float(accuracy_score(yt,pred))
    f1=float(f1_score(yt,pred,labels=LABELS,average='macro',zero_division=0))
    matrix=confusion_matrix(yt,pred,labels=LABELS)
    confusions.append(matrix)
    # No threshold selection on test animal.
    vp=rf.predict_proba(Xval)
    v_max=vp.max(axis=1)
    v_pred=rf.classes_[vp.argmax(axis=1)]
    feasible=[]
    for t in THRESHOLDS:
        keep=v_max>=t
        n=int(keep.sum())
        if n>=MIN_ACCEPTED and np.mean(v_pred[keep]==yval[keep])>=TARGET_ACC:
            feasible.append((float(keep.mean()),float(t)))
    threshold=max(feasible)[1] if feasible else 1.01
    tp=rf.predict_proba(Xt)
    accepted=tp.max(axis=1)>=threshold
    coverage=float(accepted.mean())
    accepted_acc=float(np.mean(pred[accepted]==yt[accepted])) if accepted.any() else None
    row={'test_cow':test_cow,'validation_cow':val_cow,'training_cows':','.join(map(str,train_cows)),
         'n_windows':len(yt),'n_test_classes':len(set(yt)),
         'accuracy':acc,'macro_f1_all_4_labels':f1,
         'selected_threshold':threshold,'accepted_coverage':coverage,
         'accepted_accuracy':accepted_acc,'n_accepted':int(accepted.sum())}
    records.append(row)
    p,r,f,s=precision_recall_fscore_support(yt,pred,labels=LABELS,zero_division=0)
    for j,label in enumerate(LABELS):
        per_class.append({'test_cow':test_cow,'label':label,'n_actual':int(s[j]),
                         'precision':float(p[j]),'recall':None if s[j]==0 else float(r[j]),
                         'f1':float(f[j])})
    for ytrue,ypred in zip(yt,pred):
        all_predictions.append({'test_cow':test_cow,'true_label':ytrue,'pred_label':ypred})
    print(f'Cow {test_cow}: n={len(yt)} | accuracy={acc:.1%} | macro F1={f1:.1%} | accepted={coverage:.1%}')

summary=pd.DataFrame(records)
class_df=pd.DataFrame(per_class)
pred_df=pd.DataFrame(all_predictions)
display(summary.round(4))
print('Mean accuracy across 6 held-out cows:',round(summary.accuracy.mean(),4))
print('Mean macro F1 across 6 held-out cows:',round(summary.macro_f1_all_4_labels.mean(),4))
print('Caution: 6 cows, not 6 independent farms. Window counts are not independent subjects.')

## 7 — Rumination error analysis

Check whether RUS is present and how often RUS was predicted correctly. A 0% recall or 0 test examples is not equivalent to absence of the activity in the world.

In [ ]:
rus=class_df[class_df.label=='RUS'][['test_cow','n_actual','precision','recall','f1']].copy()
display(rus)
print('RUS support across held-out tests:',int(rus.n_actual.sum()))
print('Confusions for true RUS:')
display(pd.crosstab(pred_df.loc[pred_df.true_label=='RUS','true_label'],pred_df.loc[pred_df.true_label=='RUS','pred_label'],dropna=False))
fig,ax=plt.subplots(figsize=(7.5,4))
ax.bar(summary.test_cow.astype(str),summary.accuracy*100,label='Accuracy')
ax.plot(summary.test_cow.astype(str),summary.macro_f1_all_4_labels*100,marker='o',label='Macro F1')
ax.set(xlabel='Held-out cow',ylabel='Score (%)',title='Generalization to different animals')
ax.set_ylim(0,105);ax.legend();plt.tight_layout()
plt.savefig(OUT/'cross_animal_performance.png',dpi=160);plt.show()

fig,ax=plt.subplots(figsize=(6,5))
total_cm=sum(confusions)
disp=ConfusionMatrixDisplay(total_cm,display_labels=LABELS)
disp.plot(ax=ax,cmap='Blues',values_format='d',colorbar=False)
ax.set_title('Pooled held-out predictions (all six rounds)')
plt.tight_layout();plt.savefig(OUT/'cross_animal_confusion.png',dpi=160);plt.show()

## 8 — Save results for your research portfolio

The notebook exports CSV results, charts, a JSON summary, and notes. **It does not export a replacement production model**; it compares held-out animals under different training folds. Keep your existing website model unchanged until you decide how to train and validate a replacement.

In [ ]:
from zipfile import ZipFile
summary.to_csv(OUT/'cross_animal_metrics.csv',index=False)
class_df.to_csv(OUT/'per_class_metrics.csv',index=False)
pd.DataFrame(sum(confusions)).to_csv(OUT/'pooled_confusion_counts.csv',index=False,header=False)
metadata={
 'study':'CattleSense AI v1.1 six held-out cow analysis',
 'dataset':'Japanese Black Beef Cow Behavior Classification Dataset v2',
 'dataset_url':DATA_URL,'classes':LABELS,'model':'RandomForestClassifier',
 'model_n_estimators':150,'model_max_depth':18,
 'sensor_hz':25,'window_samples':50,'stride_samples':25,
 'test_cows':ALL_COWS,'mean_accuracy_6_animals':float(summary.accuracy.mean()),
 'mean_macro_f1_6_animals':float(summary.macro_f1_all_4_labels.mean()),
 'validation_threshold_selection':'85% validation accepted accuracy, >=30 accepted; otherwise reject all',
 'note':'Averages are descriptive across six animals, not confidence intervals or a general-population estimate.'}
(OUT/'research_summary.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
(OUT/'LIMITATIONS.txt').write_text('Six Japanese Black beef cows, not a validated dairy-cow wearable.\n'
 'The dataset is not re-distributed here.\n'
 'Some behaviours are missing on individual animals. Macro F1 includes 4 target classes.\n'
 'Overlapping windows are correlated; cow is the evaluation unit.\n'
 'Model scores and selectivity thresholds may be poorly calibrated.\n'
 'No firmware, low-power measurements or new hardware sensor testing were performed.\n',encoding='utf-8')
out_zip=Path('CattleSense_v1_1_Research_Results.zip')
with ZipFile(out_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        z.write(f,arcname=f.name)
print('Created:',out_zip,'size KB:',round(out_zip.stat().st_size/1024))
try:
    from google.colab import files
    files.download(str(out_zip))
except ImportError:
    print('Download ZIP from the working directory.')

## Next stage

Upload `CattleSense_v1_1_Research_Results.zip` in our chat. We will read the actual scores **before** attempting any model improvement. Do not claim that performance improved simply because the evaluation is more extensive.